# Chapter 2 Lab — Part B: SentencePiece-Style Unigram Tokenization and Context-Window Arithmetic

**Estimated time:** 15 minutes · **Read-and-run.** Standard library only. Run Part A first (it builds `merges` and `bpe_encode`); this notebook re-creates them in the first cell so it also runs on its own.

BPE (Part A) *merges* frequent pairs bottom-up. SentencePiece's default **unigram** model starts from a large candidate vocabulary and *prunes* it, scoring each segmentation by the product of piece probabilities. Both are subword tokenizers; they differ in how they split a word they have never seen and in how they treat whitespace (SentencePiece encodes the space itself as `▁`, so it is reversible and language-agnostic).

In [ ]:
import re, collections, math

# --- Re-create the Part A mini-BPE so this notebook runs stand-alone ---
corpus = ['low lower lowest', 'newer wider', 'new news newest', 'the network learns']
def get_vocab(corpus):
    v = collections.Counter()
    for line in corpus:
        for w in line.split(): v[' '.join(list(w)) + ' </w>'] += 1
    return v
def merges_step(vocab):
    pairs = collections.Counter()
    for word, freq in vocab.items():
        sym = word.split()
        for i in range(len(sym)-1): pairs[(sym[i], sym[i+1])] += freq
    if not pairs: return None, vocab
    best = max(pairs, key=pairs.get)
    patt = re.compile(r'(?<!\S)'+re.escape(' '.join(best))+r'(?!\S)')
    return best, {patt.sub(''.join(best), w): f for w,f in vocab.items()}
vocab = get_vocab(corpus); merges = []
for _ in range(12):
    best, vocab = merges_step(vocab)
    if best is None: break
    merges.append(best)
def bpe_encode(word, merges):
    sym = list(word) + ['</w>']
    for a,b in merges:
        i=0; out=[]
        while i < len(sym):
            if i < len(sym)-1 and sym[i]==a and sym[i+1]==b: out.append(a+b); i+=2
            else: out.append(sym[i]); i+=1
        sym = out
    return sym
print('BPE ready with', len(merges), 'merges')

## A SentencePiece-style unigram tokenizer in ~30 lines

We seed the vocabulary with every substring up to length 6 that appears in the corpus, give each a probability from its frequency, and segment a word with Viterbi: the split whose pieces have the highest total log-probability wins. Real SentencePiece then prunes the vocabulary over several EM rounds; we skip pruning to keep the idea visible.

In [ ]:
def unigram_vocab(corpus, max_len=6):
    counts = collections.Counter()
    for line in corpus:
        for w in line.split():
            w = '▁' + w                      # SentencePiece marks word starts with ▁ (U+2581)
            for i in range(len(w)):
                for j in range(i+1, min(len(w), i+max_len)+1):
                    counts[w[i:j]] += 1
    total = sum(counts.values())
    return {p: math.log(c/total) for p, c in counts.items()}

def unigram_encode(word, logp, unk_penalty=-20.0):
    w = '▁' + word
    best = [(-math.inf, None)] * (len(w)+1); best[0] = (0.0, None)
    for j in range(1, len(w)+1):
        for i in range(max(0, j-6), j):
            piece = w[i:j]
            score = best[i][0] + logp.get(piece, unk_penalty if len(piece)==1 else -math.inf)
            if score > best[j][0]: best[j] = (score, i)
    pieces, j = [], len(w)
    while j > 0:
        i = best[j][1]; pieces.append(w[i:j]); j = i
    return pieces[::-1]

logp = unigram_vocab(corpus)
print('unigram candidate vocabulary size:', len(logp))
for w in ['newer', 'lowest', 'transformer', 'networking']:
    b = bpe_encode(w, merges); u = unigram_encode(w, logp)
    print(f'{w:12s}  BPE {len(b):2d} tokens {b}')
    print(f'{"":12s}  UNI {len(u):2d} tokens {u}')

Notice two things. First, both tokenizers represent `transformer` without `<UNK>`, but they split it differently — BPE follows its learned merge order, unigram picks the globally most probable segmentation. Second, the unigram pieces carry `▁`, so `''.join(pieces).replace('▁',' ')` reproduces the original text exactly; that reversibility is why SentencePiece is used for multilingual models where whitespace is not a reliable word boundary.

In [ ]:
# OPTIONAL: the real thing, only if installed (pip install sentencepiece). Trains on a tiny corpus.
try:
    import sentencepiece as spm, io, tempfile, os
    tmp = tempfile.mkdtemp(); txt = os.path.join(tmp, 'c.txt')
    with open(txt, 'w') as f: f.write('\n'.join(corpus * 20))
    spm.SentencePieceTrainer.train(input=txt, model_prefix=os.path.join(tmp,'m'), vocab_size=40, model_type='unigram', minloglevel=2)
    sp = spm.SentencePieceProcessor(model_file=os.path.join(tmp,'m.model'))
    for w in ['newer', 'transformer']:
        print(f'{w:12s} -> {sp.encode(w, out_type=str)}')
except Exception as e:
    print('sentencepiece not installed (optional):', type(e).__name__)

## From token counts to context windows and cost

The exam asks you to reason about tokens *quantitatively*: how much of a document fits, what a request costs, and how a scaling-law budget splits between parameters and data. The numbers below are illustrative — plug in your own.

In [ ]:
# Rule of thumb for English with a modern BPE vocabulary: ~1.3 tokens per word (~4 characters per token).
TOKENS_PER_WORD = 1.3
docs = {'one-page memo': 500, '10-K annual report': 60_000, 'RFC 9110 (HTTP semantics)': 95_000}
windows = {'8K': 8_192, '32K': 32_768, '128K': 131_072}
print(f'{"document":28s} {"tokens":>8s}  ' + '  '.join(f'{k:>6s}' for k in windows))
for name, words in docs.items():
    toks = int(words * TOKENS_PER_WORD)
    fits = ['  fits' if toks <= w else f'{toks/w:5.1f}x' for w in windows.values()]
    print(f'{name:28s} {toks:8,d}  ' + '  '.join(fits))

# Cost: price per 1M tokens (illustrative). Input and output are priced separately.
price_in, price_out = 0.50, 1.50          # $ per 1M tokens
prompt_tokens, answer_tokens, calls_per_day = 3_000, 400, 20_000
daily = (prompt_tokens*price_in + answer_tokens*price_out) / 1e6 * calls_per_day
print(f'\nDaily spend at {calls_per_day:,} calls: ${daily:,.2f}  -> halve the prompt (RAG trimming, caching) saves ${prompt_tokens/2*price_in/1e6*calls_per_day:,.2f}/day')

In [ ]:
# Compute-optimal scaling (Chinchilla heuristic): train on ~20 tokens per parameter.
TOKENS_PER_PARAM = 20
for params in [1e9, 7e9, 70e9]:
    tokens = params * TOKENS_PER_PARAM
    flops = 6 * params * tokens              # ~6 FLOPs per parameter per training token
    print(f'{params/1e9:5.0f}B params -> {tokens/1e9:7,.0f}B tokens, ~{flops:.1e} FLOPs')
print('\nIf your dataset is fixed at 300B tokens, the compute-optimal model is ~15B params - a bigger model is under-trained, not better.')

## Exercise (5 minutes)
1. Change `TOKENS_PER_WORD` to 2.0 (typical for code or non-English text). Which documents stop fitting in 32K?
2. Your team is offered a 70B checkpoint but only has 400B tokens of clean data. Using the cell above, argue in one sentence whether to take it or train a smaller model longer.

## Key takeaway
BPE and unigram both avoid `<UNK>` but segment unseen words differently; SentencePiece's `▁` makes tokenization reversible and language-agnostic. Scaling laws and tokenization choices determine training efficiency and effective context length — both are cost levers the exam expects you to reason about with numbers.